In [2]:
import pandas as pd
import joblib  # 변수명 리스트 저장을 위해 사용 (없으면 pip install joblib)

# ----------------------------------------------------------------------------------
# 1. 초기 설정 및 데이터 로드
# ----------------------------------------------------------------------------------
file_path = '202207_corpor_CB.csv'
df = pd.read_csv(file_path)

# 분석에 사용할 독립변수(X) 리스트 정의 (총 64개)
feature_cols = [
    'FN1_1', 'FN1_2', 'FN1_3', 'FN1_4', 'FN1_5', 'FN1_6', 'FN1_7', 'FN1_8',
    'FN1_9', 'FN1_10', 'FN1_11', 'FN1_11_1', 'FN1_11_2', 'FN1_11_3', 'FN1_11_4',
    'FN1_13', 'FN1_13_1', 'FN1_14', 'FN1_15', 'FN1_16', 'FN1_17', 'FN1_18',
    'FN1_19', 'FN1_20', 'FN1_21', 'FN1_21_1', 'FN1_22', 'FN1_22_1', 'FN1_22_2',
    'FN1_23', 'FN1_24', 'FN1_24_1', 'FN2_1', 'FN2_1_1', 'FN2_2', 'FN2_2_1',
    'FN2_3', 'FN2_3_1', 'FN2_3_2', 'FN2_3_3', 'FN2_3_4', 'FN2_3_5', 'FN2_4',
    'FN2_5', 'FN2_5_1', 'FN2_7', 'FN2_8', 'FN2_9', 'FN2_10', 'FN2_10_1',
    'FN3_1', 'FN3_2', 'FN3_2_1', 'FN3_2_2', 'FN3_3', 'FN3_4_1', 'FN3_4_2',
    'FN3_6', 'FN3_7', 'FN3_8', 'FN3_10', 'FN3_10_1', 'FN3_11', 'FN3_11_1'
]

# 식별자 및 타겟 변수
id_col = 'ID'
target_col = 'PERF_12M'

# 최종적으로 필요한 전체 컬럼 리스트
selected_cols = [id_col] + feature_cols + [target_col]

print("데이터 로드 완료:", df.shape)

# ----------------------------------------------------------------------------------
# 2. EDA: 타겟 업종 선정 (부도 건수 기준)
# ----------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("EDA: (외감구분 + 업종) 조합별 부도 현황 분석")
print("=" * 80)

# 그룹핑 및 집계
combination_default = df.groupby(['WG_GB', 'SIC_CD_3']).agg({
    target_col: ['sum', 'count', 'mean']
})
combination_default.columns = ['부도건수', '전체건수', '부도율']
combination_default = combination_default.sort_values('부도건수', ascending=False)

# 상위 1개 조합 자동 선택
top_combination = combination_default.iloc[0]
top_wg, top_industry = combination_default.index[0]

print(f"1위 조합 선정 결과:")
print(f"- 외감구분: {top_wg}")
print(f"- 업종코드: {top_industry} (도매 및 상품중개업 등)")
print(f"- 부도건수: {int(top_combination['부도건수'])}건")
print(f"- 전체건수: {int(top_combination['전체건수'])}건")
print(f"- 부도율  : {top_combination['부도율']:.2%}")

# ----------------------------------------------------------------------------------
# 3. 데이터 필터링 (전체 데이터 유지 - 언더샘플링 X)
# ----------------------------------------------------------------------------------
# 해당 조합에 맞는 데이터만 추출
final_df = df[(df['WG_GB'] == top_wg) & (df['SIC_CD_3'] == top_industry)][selected_cols].copy()

# 결측치 확인
if final_df.isnull().sum().sum() == 0:
    print("\n결측치 확인: 없음 (Clean Data)")
else:
    print("\n[주의] 결측치가 발견되었습니다. 전처리가 필요할 수 있습니다.")
    print(final_df.isnull().sum()[final_df.isnull().sum() > 0])

# ----------------------------------------------------------------------------------
# 4. 파일 저장 (CSV + PKL)
# ----------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("파일 저장 프로세스")
print("=" * 80)

# (1) 모델링용 데이터셋 저장 (CSV)
csv_filename = 'selected_data_for_modeling.csv'
final_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
print(f"1. 데이터셋 저장 완료: {csv_filename}")
print(f"   - 저장된 데이터 형상: {final_df.shape}")
print(f"   - 포함된 부도 건수: {final_df[target_col].sum()}건")

# (2) 변수명 리스트 저장 (PKL) - 중요!
# 이후 단계(Step 2, 3)에서 컬럼 순서가 바뀌거나 누락되는 것을 방지하기 위함
pkl_filename = 'feature_names.pkl'
joblib.dump(feature_cols, pkl_filename)
print(f"2. 변수명 리스트 저장 완료: {pkl_filename}")
print(f"   - 저장된 변수 개수: {len(feature_cols)}개")

print("\n[Step 1 완료] 다음 단계(Step 2_Modeling)로 넘어가세요.")

데이터 로드 완료: (150000, 109)

EDA: (외감구분 + 업종) 조합별 부도 현황 분석
1위 조합 선정 결과:
- 외감구분: 2
- 업종코드: G46 (도매 및 상품중개업 등)
- 부도건수: 266건
- 전체건수: 15482건
- 부도율  : 1.72%

결측치 확인: 없음 (Clean Data)

파일 저장 프로세스
1. 데이터셋 저장 완료: selected_data_for_modeling.csv
   - 저장된 데이터 형상: (15482, 66)
   - 포함된 부도 건수: 266건
2. 변수명 리스트 저장 완료: feature_names.pkl
   - 저장된 변수 개수: 64개

[Step 1 완료] 다음 단계(Step 2_Modeling)로 넘어가세요.
